# Stage 4.1 — masked DenseNet-121 training (issue #16)

Trains DenseNet-121 on the Stage 3 lung-masked X-rays with **exactly the Stage 2 settings**,
validating only (the test split is never touched here).

## Before you run

1. **Settings → Accelerator → GPU T4 x2** (or P100; T4 is what Stage 1/2 used).
2. **Settings → Internet → On**. Kaggle requires a **phone-verified account** for this.
3. **Add Data → Datasets → search `paultimothymooney/chest-xray-pneumonia` → Add**.

## What happens, in order

1. Clone `stage4-masking-report` (the branch contains all Stage 4 code).
2. Download the team Drive `lung_masks.zip` (7.2 MB, 5,856 masks) and check the count.
3. Build `masked_chest_xray/` from the committed split CSVs.
4. Train Phase A (head only) then Phase B (full fine-tune).
5. Write `/kaggle/working/stage4_bundle.zip` with both checkpoints and the Stage 4 results.

Expected time: smoke test ~8 min (mostly the mask download and masked-copy build), full run
~10 more minutes on a T4. The notebook keeps printing, so it will not idle out.

Run the cells top to bottom. Cell 5 is a one-epoch smoke test — run it once, read the log,
then run cell 7 for the real training.

In [ ]:
import os
import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

DATA = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
ALT = "/kaggle/input/chest-xray-pneumonia/chest_xray"
print("dataset:", DATA if os.path.isdir(DATA) else ALT if os.path.isdir(ALT) else "NOT FOUND")
assert torch.cuda.is_available(), "Turn on Settings -> Accelerator -> GPU"
assert os.path.isdir(DATA) or os.path.isdir(ALT), "Add the chest-xray-pneumonia dataset"

In [ ]:
# Clone the Stage 4 branch. Re-run this cell to pick up new commits.
import shutil
import subprocess

REPO = "/kaggle/working/stage4-repo"
REF = "stage4-masking-report"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REF,
    "https://github.com/Abhishek-Singh-2008/Pneumonia-detection.git", REPO,
], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

## Smoke test (1 epoch per phase)

Checks the settings guard, the pixel/mask alignment guard, both training phases, the
checkpoints and `summary.json`. In the log you should see:

- `Stage 2 settings match (Phase A and Phase B).`
- `Masked image check passed: 5856 images ...`
- two `Phase A/B: best epoch 1` lines

This run is **not** a result: one epoch cannot learn anything. Do not commit its numbers.

In [ ]:
import os
import subprocess
import sys

env = {**os.environ, "STAGE4_REPO": REPO, "STAGE4_SMOKE": "1"}
subprocess.run([sys.executable, f"{REPO}/kaggle/run_masked_training.py"], env=env, check=True)

## Full run

Same settings as Stage 2: Phase A 20 epochs / patience 5 / batch 32 / lr 1e-3, then
Phase B 15 / 4 / 16 / 1e-4, dropout 0.2, weight decay 1e-4, selection on validation loss.

The masked copies from the smoke test are reused; the masks are not downloaded again.

In [ ]:
import os
import subprocess
import sys

env = {**os.environ, "STAGE4_REPO": REPO, "STAGE4_SMOKE": "0"}
subprocess.run([sys.executable, f"{REPO}/kaggle/run_masked_training.py"], env=env, check=True)

## Take the results out

Download `/kaggle/working/stage4_bundle.zip` from the **Output** panel of this notebook
(or the file browser on the right). It contains:

```
densenet121_masked_phaseA_best.pth
densenet121_masked_phaseB_best.pth
results/stage4_masked_densenet/summary.json
results/stage4_masked_densenet/phaseA/{history.csv,settings.json,val_metrics.csv,val_predictions.csv}
results/stage4_masked_densenet/phaseB/{history.csv,settings.json,val_metrics.csv,val_predictions.csv}
```

Send the zip to Himanshu; the checkpoints also go to the team Drive folder. Stage 4.2
(`src/evaluate_masked.py`) needs `densenet121_masked_phaseB_best.pth`.

In [ ]:
from pathlib import Path
import zipfile

bundle = Path("/kaggle/working/stage4_bundle.zip")
print(f"{bundle}  {bundle.stat().st_size / 1e6:.1f} MB")
with zipfile.ZipFile(bundle) as zf:
    for info in zf.infolist():
        print(f"{info.file_size / 1e6:8.2f} MB  {info.filename}")